## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holidays as holidays_lib
import requests

## Helper Functions

In [2]:
def _clean_ktm_data(df:pd.DataFrame, year:str) -> pd.DataFrame:
    df = df.copy()

    ## 1. Remove ridership that is negative
    df = df[df['ridership'] > 0]

    ## 2. Remove station that is 'Unknown;
    df = df[df['origin'] != 'Unknown']

    ## 3. Remove duplicates
    df = df.drop_duplicates()

    ## 4. Remove rows with missing values
    df = df.dropna()
    
    ## 5. Make sure the year is on the specified year
    
    df = df[df['date'].dt.year == int(year)]

    return df

def get_clean_ktm_data(year:int)->pd.DataFrame:
    """
    Get Official KTMB data from 'https://storage.data.gov.my/transportation/ktmb/{year}.parquet'
    """
    
    URL_DATA = f'https://storage.data.gov.my/transportation/ktmb/komuter_{year}.parquet'
    
    # Read data
    df = pd.read_parquet(URL_DATA, engine='fastparquet')
    
    # Cast datetime format
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        
    return _clean_ktm_data(df, year)

In [3]:
def get_weather_data(weather_features: list) -> pd.DataFrame:
    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude": 3.139, "longitude": 101.6869,          # Kuala Lumpur
            "start_date": "2024-01-01", "end_date": "2024-12-31",
            "hourly": ",".join(weather_features),
            "timezone": "Asia/Kuala_Lumpur",
        },
        timeout=60,
    )
    response.raise_for_status()
    df =  pd.DataFrame(response.json()["hourly"]).rename(columns={'time':'datetime'})
    
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    return df


def get_temporal_data(year:int)->pd.DataFrame:
    YEAR_HOURS = pd.date_range(f"{year}-01-01", f"{year}-12-31 23:00", freq="h")
    
    ### First-level (base)
    my_holidays = holidays_lib.Malaysia(years=sorted(set(YEAR_HOURS.year)), subdiv="KUL")
    day_of_week = YEAR_HOURS.dayofweek.to_numpy()
    hour = YEAR_HOURS.hour.to_numpy()

    ### Second-levek (derrived from first-level) 
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    dow_sin = np.sin(2 * np.pi * day_of_week / 7)
    dow_cos = np.cos(2 * np.pi * day_of_week / 7)
    is_weekend = (day_of_week >= 5).astype(float)
    is_holiday = np.array([d in my_holidays for d in YEAR_HOURS.date], dtype=float)
    
    column_name = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_holiday']
    
    np_stack = np.stack([
        hour_sin, hour_cos, dow_sin, dow_cos, is_weekend, is_holiday,
    ], axis=1)
    
    return pd.DataFrame(np_stack, columns=column_name, index=YEAR_HOURS).reset_index().rename(columns={'index':'datetime'})

In [4]:
def join_all_df(ktm_df:pd.DataFrame, weather_df:pd.DataFrame, temporal_df:pd.DataFrame)->pd.DataFrame:
    
    selected_cols = ['datetime', 'origin', 'destination', 'ridership']
        
    ktm_df['od'] = ktm_df['origin'] + '_' + ktm_df['destination']

    ktm_df['datetime'] = ktm_df['date'] + pd.to_timedelta(ktm_df['time'] + ':00')
    
    
    ktm_df = ktm_df[selected_cols]
    
    joined_df = ktm_df.merge(
        weather_df, on='datetime', how='left'
    ).merge(
        temporal_df, on='datetime', how='left'
    )
    
    return joined_df

In [ ]:
def build_station_series(
    ktm_df:pd.DataFrame,
    datetime_index:pd.DatetimeIndex,
    stations:list
) -> np.ndarray:
    
    """
    Convert KTM tabular data to numpy format.
    """
    
    agg_df = ktm_df.groupby([ktm_df['datetime'].dt.floor('h'), 'origin'])['ridership'].sum()
    
    # Pivot from long format to wide format
    # To fill 0 for stations without ridership at certain datetime
    # whereby other stations have data
    pivot_df = agg_df.unstack(fill_value=0) 
    
    # Reindex based on intialised value upstream (to ensure parity)
    # To fill 0 for stations that never appear as origin
    # or datetime (index) that has no data at all
    pivot_df = pivot_df.reindex(index=datetime_index, columns=stations, fill_value=0)
    
    # Convert to numpy
    numpy_df = pivot_df.to_numpy(dtype=np.float32)
    
    return numpy_df

In [60]:
def build_od_flow(ktm_df:pd.DataFrame,
                  STATIONS_LIST:list[str]
) -> np.ndarray :
    """
    Aggregate the daily tabular KTM data
    into a total annual ridership per origin-destination.
    
    Output format is {'origin: destination'}, ridership
    
    """
    
    # Obtain annual ridership for each origin-destination pair
    od_agg_df = ktm_df.groupby(['origin', 'destination'])['ridership'].sum()
    
    # Establish station index dict
    station_idx = {s:i for i, s in enumerate(STATIONS_LIST)}
    
    # Initialise empty numpy array to store the OD ridership
    od_flow = np.ndarray((len(STATIONS_LIST), len(STATIONS_LIST)), dtype=np.float32)
    
    for (o,d), ridership in od_agg_df.items():
        od_flow[station_idx[o], station_idx[d]] = ridership
    
    return od_flow + od_flow.T

In [81]:
def convert_df_to_numpy(df:pd.DataFrame, cols_to_drop:list[str])->np.ndarray:
    
    return df.drop(columns=cols_to_drop).to_numpy()

In [61]:
def build_adjacency_matrix(od_flow:np.ndarray, threshold:int)->np.ndarray:
    """
    Build Origin-Destination (OD) adjacency matrix.

    If the annual sum of an OD exceeds the defined threshold,
    we consider that the stations are connected.
    
    We represent this connection through a matrix of Origin station (row-level)
    to Destination station (column-level).
    
    If the connection is present, we fill an integer 1, else 0.
    """
    
    return (od_flow >= threshold).astype(np.float32)

## [1] Data Collection and Cleaning

### 1.1 KTM Data

In [6]:
ktm_2024_df = get_clean_ktm_data(2024)

In [7]:
ktm_2024_df

,date,time,origin,destination,ridership
0,2024-01-01,00:00,Subang Jaya,Padang Jawa,2
1,2024-01-01,00:00,Kajang,Nilai,3
2,2024-01-01,00:00,KL Sentral,Nilai,10
3,2024-01-01,00:00,KL Sentral,Bukit Badak,6
4,2024-01-01,00:00,KL Sentral,Klang,16
...,...,...,...,...,...
2686103,2024-12-31,23:00,Midvalley,Sungai Gadut,1
2686104,2024-12-31,23:00,Subang Jaya,Padang Jawa,8
2686105,2024-12-31,23:00,Abdullah Hukum,Padang Jawa,1
2686106,2024-12-31,23:00,KL Sentral,Padang Jawa,6


In [8]:
## Derrive the stations and index in the beginning

STATIONS_LIST = sorted(set(ktm_2024_df['origin']) | set(ktm_2024_df['destination']))
DT_INDEX = pd.date_range("2024-01-01", "2024-12-31 23:00", freq="h")

### 1.2 Weather Data

In [9]:
weather_features = [
    'temperature_2m',
    'relative_humidity_2m',
    'precipitation'
]

weather_df = get_weather_data(weather_features)

In [10]:
weather_df

,datetime,temperature_2m,relative_humidity_2m,precipitation
0,2024-01-01 00:00:00,25.6,95,0.0
1,2024-01-01 01:00:00,25.4,96,0.0
2,2024-01-01 02:00:00,25.1,96,0.0
3,2024-01-01 03:00:00,24.8,98,0.0
4,2024-01-01 04:00:00,24.7,97,0.0
...,...,...,...,...
8779,2024-12-31 19:00:00,26.1,88,0.1
8780,2024-12-31 20:00:00,25.7,93,0.0
8781,2024-12-31 21:00:00,25.9,95,0.2
8782,2024-12-31 22:00:00,25.5,96,1.0


### 1.3 Temporal Data

In [11]:
temporal_df = get_temporal_data(2024)

In [12]:
temporal_df

,datetime,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend,is_holiday
0,2024-01-01 00:00:00,0.000000,1.000000,0.000000,1.00000,0.0,1.0
1,2024-01-01 01:00:00,0.258819,0.965926,0.000000,1.00000,0.0,1.0
2,2024-01-01 02:00:00,0.500000,0.866025,0.000000,1.00000,0.0,1.0
3,2024-01-01 03:00:00,0.707107,0.707107,0.000000,1.00000,0.0,1.0
4,2024-01-01 04:00:00,0.866025,0.500000,0.000000,1.00000,0.0,1.0
...,...,...,...,...,...,...,...
8779,2024-12-31 19:00:00,-0.965926,0.258819,0.781831,0.62349,0.0,0.0
8780,2024-12-31 20:00:00,-0.866025,0.500000,0.781831,0.62349,0.0,0.0
8781,2024-12-31 21:00:00,-0.707107,0.707107,0.781831,0.62349,0.0,0.0
8782,2024-12-31 22:00:00,-0.500000,0.866025,0.781831,0.62349,0.0,0.0


In [13]:
join_all_df(ktm_2024_df, weather_df, temporal_df)

,datetime,origin,destination,ridership,temperature_2m,relative_humidity_2m,precipitation,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend,is_holiday
0,2024-01-01 00:00:00,Subang Jaya,Padang Jawa,2,25.6,95,0.0,0.000000,1.000000,0.000000,1.00000,0.0,1.0
1,2024-01-01 00:00:00,Kajang,Nilai,3,25.6,95,0.0,0.000000,1.000000,0.000000,1.00000,0.0,1.0
2,2024-01-01 00:00:00,KL Sentral,Nilai,10,25.6,95,0.0,0.000000,1.000000,0.000000,1.00000,0.0,1.0
3,2024-01-01 00:00:00,KL Sentral,Bukit Badak,6,25.6,95,0.0,0.000000,1.000000,0.000000,1.00000,0.0,1.0
4,2024-01-01 00:00:00,KL Sentral,Klang,16,25.6,95,0.0,0.000000,1.000000,0.000000,1.00000,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2674284,2024-12-31 23:00:00,Midvalley,Sungai Gadut,1,24.9,98,0.0,-0.258819,0.965926,0.781831,0.62349,0.0,0.0
2674285,2024-12-31 23:00:00,Subang Jaya,Padang Jawa,8,24.9,98,0.0,-0.258819,0.965926,0.781831,0.62349,0.0,0.0
2674286,2024-12-31 23:00:00,Abdullah Hukum,Padang Jawa,1,24.9,98,0.0,-0.258819,0.965926,0.781831,0.62349,0.0,0.0
2674287,2024-12-31 23:00:00,KL Sentral,Padang Jawa,6,24.9,98,0.0,-0.258819,0.965926,0.781831,0.62349,0.0,0.0


In [73]:
weather_df

,datetime,temperature_2m,relative_humidity_2m,precipitation
0,2024-01-01 00:00:00,25.6,95,0.0
1,2024-01-01 01:00:00,25.4,96,0.0
2,2024-01-01 02:00:00,25.1,96,0.0
3,2024-01-01 03:00:00,24.8,98,0.0
4,2024-01-01 04:00:00,24.7,97,0.0
...,...,...,...,...
8779,2024-12-31 19:00:00,26.1,88,0.1
8780,2024-12-31 20:00:00,25.7,93,0.0
8781,2024-12-31 21:00:00,25.9,95,0.2
8782,2024-12-31 22:00:00,25.5,96,1.0


In [85]:
convert_df_to_numpy(weather_df, ['datetime']).shape

(8784, 3)

In [84]:
convert_df_to_numpy(temporal_df, ['datetime']).shape

(8784, 6)

## [2] Data Preprocessing

### 2.1 Data Cleaning

### 2.2 Data Transformation and Integration

### 2.3 Feature Engineering

### 2.4 Graph Structure

In [14]:
hour_origin_ridership_df = ktm_2024_df.groupby([ktm_2024_df['datetime'].dt.floor("h"), "origin"])['ridership'].sum().reset_index()

In [15]:
(
    ktm_2024_df.groupby([ktm_2024_df['datetime'].dt.floor("h"), "origin"])['ridership'].sum().reset_index()
)


,datetime,origin,ridership
0,2024-01-01 00:00:00,Bandar Tasek Selatan,5
1,2024-01-01 00:00:00,Bangi,1
2,2024-01-01 00:00:00,Batu Tiga,1
3,2024-01-01 00:00:00,KL Sentral,56
4,2024-01-01 00:00:00,Kajang,9
...,...,...,...
367030,2024-12-31 23:00:00,Subang Jaya,14
367031,2024-12-31 23:00:00,Sungai Buloh,19
367032,2024-12-31 23:00:00,Taman Wahyu,1
367033,2024-12-31 23:00:00,Tanjong Malim,3


In [16]:
(
    ktm_2024_df.groupby([ktm_2024_df['datetime'].dt.floor("h"), "origin"])['ridership'].sum()
    .unstack(fill_value=0)
)


origin,Abdullah Hukum,Angkasapuri,Bandar Tasek Selatan,Bangi,Bank Negara,Batang Benar,Batang Kali,Batu Caves,Batu Kentonmen,Batu Tiga,...,Shah Alam,Subang Jaya,Sungai Buloh,Sungai Gadut,Taman Wahyu,Tanjong Malim,Telok Gadong,Telok Pulai,Tiroi,UKM
datetime,,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00,0,0,5,1,0,0,0,0,0,1,...,0,13,37,0,1,10,0,0,0,1
2024-01-01 01:00:00,0,0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2024-01-01 02:00:00,0,0,3,0,1,0,1,20,0,1,...,0,2,1,4,0,3,1,0,0,0
2024-01-01 03:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2024-01-01 04:00:00,6,0,89,38,4,30,8,23,1,10,...,33,23,36,15,3,128,15,7,7,65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00,107,3,150,45,114,24,18,77,3,69,...,58,456,216,24,9,45,38,22,4,64
2024-12-31 20:00:00,33,0,65,24,46,46,17,66,1,21,...,29,113,107,7,15,108,19,11,0,35
2024-12-31 21:00:00,24,1,63,41,15,68,12,44,2,24,...,27,68,61,11,19,48,24,5,8,19


In [17]:
(
    ktm_2024_df.groupby([ktm_2024_df['datetime'].dt.floor("h"), "origin"])['ridership'].sum()
    .unstack(fill_value=0)
    .reindex(index=DT_INDEX, columns=STATIONS_LIST, fill_value=0)
)

origin,Abdullah Hukum,Angkasapuri,Bandar Tasek Selatan,Bangi,Bank Negara,Batang Benar,Batang Kali,Batu Caves,Batu Kentonmen,Batu Tiga,...,Subang Jaya,Sungai Buloh,Sungai Gadut,Taman Wahyu,Tanjong Malim,Telok Gadong,Telok Pulai,Tiroi,UKM,Unknown
2024-01-01 00:00:00,0,0,5,1,0,0,0,0,0,1,...,13,37,0,1,10,0,0,0,1,0
2024-01-01 01:00:00,0,0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2024-01-01 02:00:00,0,0,3,0,1,0,1,20,0,1,...,2,1,4,0,3,1,0,0,0,0
2024-01-01 03:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2024-01-01 04:00:00,6,0,89,38,4,30,8,23,1,10,...,23,36,15,3,128,15,7,7,65,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00,107,3,150,45,114,24,18,77,3,69,...,456,216,24,9,45,38,22,4,64,0
2024-12-31 20:00:00,33,0,65,24,46,46,17,66,1,21,...,113,107,7,15,108,19,11,0,35,0
2024-12-31 21:00:00,24,1,63,41,15,68,12,44,2,24,...,68,61,11,19,48,24,5,8,19,0
2024-12-31 22:00:00,21,4,40,13,13,32,7,12,0,18,...,61,24,4,9,42,4,5,1,9,0


In [18]:
(
    ktm_2024_df.groupby([ktm_2024_df['datetime'].dt.floor("h"), "origin"])['ridership'].sum()
    .unstack(fill_value=0) # To fill gaps in the observed data
    .reindex(index=DT_INDEX, columns=STATIONS_LIST, fill_value=0) # To fill rows and columns that were never present in the data
).to_numpy(dtype=np.float32)

array([[ 0.,  0.,  5., ...,  0.,  1.,  0.],
       [ 0.,  0.,  1., ...,  0.,  1.,  0.],
       [ 0.,  0.,  3., ...,  0.,  0.,  0.],
       ...,
       [24.,  1., 63., ...,  8., 19.,  0.],
       [21.,  4., 40., ...,  1.,  9.,  0.],
       [25.,  0., 25., ...,  0.,  4.,  0.]], dtype=float32)

In [19]:
test = (
    ktm_2024_df.groupby([ktm_2024_df['datetime'].dt.floor("h"), "origin"])['ridership'].sum()
    .unstack(fill_value=0) # To fill gaps in the observed data
    .reindex(index=DT_INDEX, columns=STATIONS_LIST, fill_value=0) # To fill rows and columns that were never present in the data
).to_numpy(dtype=np.float32)

In [20]:
test.shape

(8784, 59)

In [21]:
test[:, STATIONS_LIST.index('Nilai')].shape

(8784,)

In [22]:
hour_origin_ridership_df

,datetime,origin,ridership
0,2024-01-01 00:00:00,Bandar Tasek Selatan,5
1,2024-01-01 00:00:00,Bangi,1
2,2024-01-01 00:00:00,Batu Tiga,1
3,2024-01-01 00:00:00,KL Sentral,56
4,2024-01-01 00:00:00,Kajang,9
...,...,...,...
367030,2024-12-31 23:00:00,Subang Jaya,14
367031,2024-12-31 23:00:00,Sungai Buloh,19
367032,2024-12-31 23:00:00,Taman Wahyu,1
367033,2024-12-31 23:00:00,Tanjong Malim,3


In [23]:
## Create a matrix with the OD flow

# Initialise empty numpy matrix

flow = np.zeros((len(STATIONS_LIST), len(STATIONS_LIST)), dtype=np.float64)

# Origin-Destination Annual Aggregated Ridership, so one row per station
od_flow_df = ktm_2024_df.groupby(['origin', 'destination'])['ridership'].sum()

station_idx = {s: i for i, s in enumerate(STATIONS_LIST)}

for (o,d), ridership in od_flow_df.items():
    flow[station_idx[o], station_idx[d]] = ridership

system_flow = flow + flow.T  # undirected: total flow either direction

In [55]:
od_flow_df

origin          destination         
Abdullah Hukum  Abdullah Hukum           508
                Angkasapuri             1209
                Bandar Tasek Selatan      96
                Bangi                     38
                Bank Negara              265
                                        ... 
UKM             Telok Gadong             148
                Telok Pulai              255
                Tiroi                    985
                UKM                      241
                Unknown                   91
Name: ridership, Length: 3332, dtype: int64

In [59]:
system_flow[:, station_idx['Midvalley']]

array([1.12000e+02, 3.00000e+01, 1.34316e+05, 4.04840e+04, 4.03500e+03,
       1.76980e+04, 6.49000e+02, 3.84780e+04, 3.94400e+03, 3.24000e+02,
       1.60000e+02, 2.80000e+01, 3.00000e+01, 8.55640e+04, 6.48540e+04,
       2.88530e+04, 2.40860e+04, 3.00000e+02, 8.50000e+01, 1.33900e+03,
       1.31600e+03, 1.01800e+03, 2.12000e+02, 5.15800e+03, 3.52000e+02,
       5.94400e+03, 5.32000e+02, 2.89570e+04, 5.34000e+02, 2.45000e+02,
       3.02000e+02, 3.90000e+01, 2.31000e+02, 7.04500e+03, 9.99800e+03,
       2.83000e+02, 2.06900e+03, 3.04100e+03, 4.33750e+04, 1.14300e+03,
       2.07950e+04, 9.20800e+03, 5.35800e+03, 8.98650e+04, 6.58630e+04,
       4.51000e+02, 1.83000e+02, 3.39000e+02, 7.48000e+02, 2.48000e+02,
       9.23000e+02, 8.77900e+03, 1.90090e+04, 6.42000e+02, 1.81000e+02,
       1.25000e+02, 1.18900e+04, 3.03240e+04, 1.87000e+02])

In [26]:
system_flow

array([[1.0160e+03, 2.1850e+03, 2.4900e+02, ..., 4.9000e+01, 7.7000e+01,
        1.6400e+02],
       [2.1850e+03, 5.6000e+01, 2.1400e+02, ..., 1.2000e+01, 1.0600e+02,
        6.0000e+00],
       [2.4900e+02, 2.1400e+02, 1.5060e+03, ..., 1.2231e+04, 7.1773e+04,
        2.5600e+02],
       ...,
       [4.9000e+01, 1.2000e+01, 1.2231e+04, ..., 1.1800e+02, 1.8760e+03,
        2.7000e+01],
       [7.7000e+01, 1.0600e+02, 7.1773e+04, ..., 1.8760e+03, 4.8200e+02,
        9.1000e+01],
       [1.6400e+02, 6.0000e+00, 2.5600e+02, ..., 2.7000e+01, 9.1000e+01,
        0.0000e+00]])

In [27]:
display((system_flow > 20).sum()) # scalar

display((system_flow > 20).sum(0)) # total per column (sum down each column)

display((system_flow > 20).sum(1)) # total per row (sum across each row)

3170

array([56, 42, 59, 59, 58, 55, 53, 59, 42, 58, 57, 46, 45, 59, 59, 55, 53,
       57, 54, 50, 55, 59, 50, 59, 50, 50, 59, 59, 58, 55, 57, 19, 56, 56,
       58, 47, 59, 47, 53, 56, 55, 57, 37, 59, 59, 52, 56, 58, 58, 56, 58,
       55, 51, 57, 56, 56, 46, 58, 48])

array([56, 42, 59, 59, 58, 55, 53, 59, 42, 58, 57, 46, 45, 59, 59, 55, 53,
       57, 54, 50, 55, 59, 50, 59, 50, 50, 59, 59, 58, 55, 57, 19, 56, 56,
       58, 47, 59, 47, 53, 56, 55, 57, 37, 59, 59, 52, 56, 58, 58, 56, 58,
       55, 51, 57, 56, 56, 46, 58, 48])

In [29]:
display((system_flow > 20).sum()) # scalar

display((system_flow > 20).sum(0).mean()) # total per column (sum down each column)

display((system_flow > 20).sum(1).mean()) # total per row (sum across each row)

3170

53.728813559322035

53.728813559322035

In [40]:
thresholds = list(np.logspace(1, 6, 40))
average_deg = [(system_flow > t).sum(1).mean() for t in thresholds]

pd.DataFrame({
    'thresholds': thresholds,
    'average_deg': average_deg
}).round(2)

,thresholds,average_deg
0,10.00,55.66
1,13.43,55.05
2,18.05,54.00
3,24.24,53.02
4,32.57,51.63
5,43.75,49.47
6,58.78,47.47
7,78.97,45.42
8,106.08,43.02
9,142.51,40.42


In [ ]:
adj_matrix =(system_flow >= 20).astype(np.float32) # convert boolean to 1 and 0

adj_matrix

array([[1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 1., 1., 1.],
       ...,
       [1., 0., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 0., 1., ..., 1., 1., 0.]], dtype=float32)

In [46]:
adj_matrix.shape

(59, 59)

In [49]:
np.fill_diagonal(adj_matrix, 0.0) # To remove self-loop , so we set the diagonal (Station A to Station A itself) to 0, not connected

In [50]:
adj_matrix

array([[0., 1., 1., ..., 1., 1., 1.],
       [1., 0., 1., ..., 0., 1., 0.],
       [1., 1., 0., ..., 1., 1., 1.],
       ...,
       [1., 0., 1., ..., 0., 1., 1.],
       [1., 1., 1., ..., 1., 0., 1.],
       [1., 0., 1., ..., 1., 1., 0.]], dtype=float32)

In [62]:
od_flow = build_od_flow(ktm_2024_df,STATIONS_LIST)
adj_matrix = build_adjacency_matrix(od_flow, 5_000)

In [ ]:
display(adj_matrix.sum()/2) # Edges

display(adj_matrix.sum(1).mean()) # Average degree, means on average how many destination station is one origination station connected to

295.5

10.016949

In [67]:
adj_matrix.sum(1)

array([19.,  0., 22., 10., 25.,  9.,  6., 22.,  2., 11.,  6.,  4.,  2.,
       56., 18.,  5.,  6.,  7.,  5.,  5.,  5., 19.,  5., 27.,  5.,  5.,
       23., 16.,  7.,  6.,  6.,  0.,  7.,  9.,  8.,  5., 15.,  4.,  3.,
        2.,  9.,  4.,  2., 14., 22.,  6.,  6., 15., 15., 16.,  9.,  9.,
        7., 11.,  6.,  6.,  5., 12.,  0.], dtype=float32)

In [72]:
adj_matrix[:, station_idx['Angkasapuri']]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

In [70]:
adj_matrix

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 1., 0.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

## [3] Model Development

### Train-test-validation Split

### M1 Variation

### M2 Variation

### M3 Variation

## [4] Model Evaluation